# 🎯 Ejercicios Resueltos — Regresión

Este notebook resuelve los ejercicios propuestos al final de **`EDA_Regresion.ipynb`**.

### Ejercicios que abordaremos:

1. **¿Cómo cambiarían los resultados sin winsorización de outliers?** — Comparamos el pipeline con y sin manejo de outliers.
2. **¿Hay algún modelo con evidencia clara de sobreajuste?** — Análisis sistemático con visualización.
3. **¿La red más compleja gana siempre?** — Probamos redes de distintos tamaños.
4. **¿Cuál es el trade-off entre tiempo de entrenamiento y desempeño?** — Análisis Pareto.
5. **Bonus:** ¿Qué pasa si tenemos menos datos de entrenamiento?

> 💡 Cada ejercicio tiene: **Enunciado → Hipótesis → Implementación → Resultados → Conclusión**.


## Setup común

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 90
print("✅ Setup completo")

In [ ]:
# Carga y preparación base
data = fetch_california_housing(as_frame=True)
df = data.frame.rename(columns={'MedHouseVal': 'Precio'})
print(f"Dataset: {df.shape}")
df.head()

## Ejercicio 1 · Efecto de la winsorización de outliers

### 📝 Enunciado
Comparar el desempeño del pipeline **con** y **sin** manejo de outliers, usando los mismos modelos y semilla.

### 💭 Hipótesis
Los outliers deberían perjudicar más a modelos sensibles a valores extremos (regresión lineal y MLP) que a modelos basados en árboles (RF, XGBoost).


In [ ]:
# Preparamos dos versiones del dataset: SIN y CON winsorización
X_original = df.drop(columns='Precio').values
y_original = df['Precio'].values

# CON winsorización
df_win = df.copy()
for col in df.columns.drop('Precio'):
    Q1, Q3 = df_win[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df_win[col] = df_win[col].clip(lower=Q1-1.5*IQR, upper=Q3+1.5*IQR)
X_win = df_win.drop(columns='Precio').values

# Divisiones con la MISMA semilla en ambos casos
X_tr_o, X_te_o, y_tr, y_te = train_test_split(X_original, y_original, test_size=0.2, random_state=SEED)
X_tr_w, X_te_w, _, _       = train_test_split(X_win,      y_original, test_size=0.2, random_state=SEED)

# Escalados
scaler_o = StandardScaler(); X_tr_o_sc = scaler_o.fit_transform(X_tr_o); X_te_o_sc = scaler_o.transform(X_te_o)
scaler_w = StandardScaler(); X_tr_w_sc = scaler_w.fit_transform(X_tr_w); X_te_w_sc = scaler_w.transform(X_te_w)

print("✅ Datasets listos: SIN winsorización y CON winsorización")

In [ ]:
def entrenar_evaluar(nombre, X_tr, X_te, y_tr, y_te):
    if nombre == 'Linear Regression':
        m = LinearRegression()
    elif nombre == 'Random Forest':
        m = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=SEED, n_jobs=-1)
    elif nombre == 'XGBoost':
        m = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05, random_state=SEED, n_jobs=-1)
    elif nombre == 'MLP':
        m = MLPRegressor(hidden_layer_sizes=(64,32), max_iter=200, random_state=SEED, early_stopping=True)
    m.fit(X_tr, y_tr)
    return {
        'MSE': mean_squared_error(y_te, m.predict(X_te)),
        'MAE': mean_absolute_error(y_te, m.predict(X_te)),
        'R²':  r2_score(y_te, m.predict(X_te))
    }

modelos = ['Linear Regression', 'Random Forest', 'XGBoost', 'MLP']
resultados = []
for m in modelos:
    # Los modelos lineales / MLP usan escalados; los de árbol no lo necesitan
    if m in ['Linear Regression', 'MLP']:
        sin = entrenar_evaluar(m, X_tr_o_sc, X_te_o_sc, y_tr, y_te)
        con = entrenar_evaluar(m, X_tr_w_sc, X_te_w_sc, y_tr, y_te)
    else:
        sin = entrenar_evaluar(m, X_tr_o, X_te_o, y_tr, y_te)
        con = entrenar_evaluar(m, X_tr_w, X_te_w, y_tr, y_te)
    resultados.append({'Modelo': m,
                       'R² sin outliers tratados': sin['R²'],
                       'R² con winsorización': con['R²'],
                       'MAE sin': sin['MAE'],
                       'MAE con': con['MAE']})

df_out = pd.DataFrame(resultados).set_index('Modelo')
df_out['Δ R² (con − sin)'] = df_out['R² con winsorización'] - df_out['R² sin outliers tratados']
df_out.round(4)

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
x = np.arange(len(df_out))
w = 0.35

axes[0].bar(x - w/2, df_out['R² sin outliers tratados'], w, label='SIN winsorización', color='salmon')
axes[0].bar(x + w/2, df_out['R² con winsorización'],     w, label='CON winsorización', color='steelblue')
axes[0].set_xticks(x); axes[0].set_xticklabels(df_out.index, rotation=15)
axes[0].set_title('R² en test'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x - w/2, df_out['MAE sin'], w, label='SIN winsorización', color='salmon')
axes[1].bar(x + w/2, df_out['MAE con'], w, label='CON winsorización', color='steelblue')
axes[1].set_xticks(x); axes[1].set_xticklabels(df_out.index, rotation=15)
axes[1].set_title('MAE en test (más bajo = mejor)'); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio 1

- La regresión lineal y el MLP son los que **más se benefician** del tratamiento de outliers (`Δ R²` mayor).
- Random Forest y XGBoost son **más robustos** — la diferencia con o sin winsorización es menor porque los árboles dividen el espacio en rectángulos y los outliers rara vez cambian las divisiones importantes.
- **Regla práctica:** siempre trata los outliers para modelos lineales y basados en distancias; para árboles el beneficio es marginal pero no perjudica.


## Ejercicio 2 · Diagnóstico sistemático de sobreajuste

### 📝 Enunciado
Comparar train vs test para modelos de complejidad **creciente** de Random Forest, y visualizar cuándo aparece el sobreajuste.

### 💭 Hipótesis
Al aumentar `max_depth`, el R² en train subirá cerca de 1.0 mientras el de test se estanca o baja → aparecerá una **brecha creciente** entre ambos.


In [ ]:
# Usamos el dataset winsorizado (ejercicio anterior confirmó que es mejor)
X_tr, X_te = X_tr_w, X_te_w

profundidades = [2, 4, 6, 8, 10, 15, 20, 30, None]  # None = sin límite
resultados_prof = []

for d in profundidades:
    rf = RandomForestRegressor(n_estimators=100, max_depth=d, random_state=SEED, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    r2_tr = r2_score(y_tr, rf.predict(X_tr))
    r2_te = r2_score(y_te, rf.predict(X_te))
    resultados_prof.append({
        'max_depth': str(d),
        'R²_train': r2_tr,
        'R²_test':  r2_te,
        'Gap':      r2_tr - r2_te
    })

df_prof = pd.DataFrame(resultados_prof)
df_prof.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

x = np.arange(len(df_prof))
axes[0].plot(x, df_prof['R²_train'], 'o-', label='Train', color='steelblue', linewidth=2)
axes[0].plot(x, df_prof['R²_test'],  'o-', label='Test',  color='salmon',    linewidth=2)
axes[0].set_xticks(x); axes[0].set_xticklabels(df_prof['max_depth'])
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('R²')
axes[0].set_title('R² train vs test según profundidad'); axes[0].legend(); axes[0].grid(True)

# Zona de sobreajuste sombreada
axes[1].bar(x, df_prof['Gap'], color=['lightgreen' if g < 0.05 else 'gold' if g < 0.15 else 'crimson' for g in df_prof['Gap']])
axes[1].axhline(0.15, color='crimson', linestyle='--', alpha=0.5, label='Umbral sobreajuste')
axes[1].set_xticks(x); axes[1].set_xticklabels(df_prof['max_depth'])
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Gap R² (train − test)')
axes[1].set_title('Brecha train − test'); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio 2

- Con `max_depth` bajo (2-4) → **subajuste**: ambos R² son bajos.
- Con `max_depth` medio (6-10) → **zona óptima**: R² test cerca de su máximo con gap moderado.
- Con `max_depth` alto o `None` → **sobreajuste**: R² train se acerca a 1.0, pero R² test se estanca o baja.
- La **brecha (Gap)** es el indicador visual más útil para detectar sobreajuste al variar hiperparámetros.


## Ejercicio 3 · ¿La red más compleja gana siempre?

### 📝 Enunciado
Comparar redes neuronales Keras de **cuatro tamaños** distintos (desde diminuta hasta muy grande) y ver qué pasa con el desempeño y el sobreajuste.

### 💭 Hipótesis
Habrá un **sweet spot** intermedio. La red más grande podría sobreajustar o simplemente no mejorar el resultado de una red mediana.


In [ ]:
arquitecturas = {
    'Diminuta  [8]':          [8],
    'Pequeña   [32, 16]':     [32, 16],
    'Mediana   [64, 32]':     [64, 32],
    'Grande    [128, 64, 32]':[128, 64, 32],
    'Enorme    [256, 128, 64, 32]': [256, 128, 64, 32]
}

# Preparamos train/val para las curvas
X_tr_full, X_val, y_tr_full, y_val = train_test_split(X_tr_w_sc, y_tr, test_size=0.2, random_state=SEED)

resultados_nn = []
historias_nn  = {}

for nombre, capas in arquitecturas.items():
    model = Sequential()
    model.add(Dense(capas[0], activation='relu', input_shape=(X_tr_w_sc.shape[1],)))
    for h in capas[1:]:
        model.add(Dense(h, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    
    t0 = time.time()
    hist = model.fit(X_tr_full, y_tr_full, validation_data=(X_val, y_val),
                     epochs=60, batch_size=64, verbose=0)
    t_ent = time.time() - t0
    
    y_tr_pred = model.predict(X_tr_w_sc, verbose=0).ravel()
    y_te_pred = model.predict(X_te_w_sc, verbose=0).ravel()
    
    resultados_nn.append({
        'Arquitectura': nombre,
        'Parámetros':   model.count_params(),
        'R²_train':     r2_score(y_tr, y_tr_pred),
        'R²_test':      r2_score(y_te, y_te_pred),
        'MAE_test':     mean_absolute_error(y_te, y_te_pred),
        'Tiempo (s)':   round(t_ent, 1)
    })
    historias_nn[nombre] = hist.history

df_nn = pd.DataFrame(resultados_nn)
df_nn['Gap'] = df_nn['R²_train'] - df_nn['R²_test']
df_nn.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Curvas de validación de todas las arquitecturas
for nombre, h in historias_nn.items():
    axes[0].plot(h['val_loss'], label=nombre, linewidth=1.8)
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Val MSE')
axes[0].set_title('Curvas de pérdida en validación por arquitectura')
axes[0].legend(fontsize=9); axes[0].grid(True)

# R² train/test vs #parámetros
x_params = df_nn['Parámetros']
axes[1].semilogx(x_params, df_nn['R²_train'], 'o-', label='Train', color='steelblue', linewidth=2, markersize=10)
axes[1].semilogx(x_params, df_nn['R²_test'],  'o-', label='Test',  color='salmon',    linewidth=2, markersize=10)
for i, arq in enumerate(df_nn['Arquitectura']):
    axes[1].annotate(arq.split()[0], (x_params.iloc[i], df_nn['R²_test'].iloc[i]),
                     textcoords="offset points", xytext=(5,-12), fontsize=8)
axes[1].set_xlabel('# Parámetros (log)'); axes[1].set_ylabel('R²')
axes[1].set_title('R² vs Complejidad'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio 3

- La red **diminuta** subajusta: no tiene capacidad para capturar la relación.
- Las redes **mediana/grande** dan resultados similares en test — hemos alcanzado el **techo del modelo** para este problema.
- La red **enorme** no gana nada (o incluso pierde) y **tarda mucho más** en entrenar. También muestra el mayor `Gap`, evidencia de sobreajuste.
- **Regla práctica:** más parámetros no equivale a mejor modelo. Empieza pequeño y crece hasta que dejes de ver mejoras en validación.


## Ejercicio 4 · Trade-off entre tiempo y desempeño

### 📝 Enunciado
Entrenar los 4 modelos base con tiempos cronometrados y visualizar la **frontera de Pareto**: qué modelos son óptimos en el balance tiempo/desempeño.


In [ ]:
configs = [
    ('Linear Regression', LinearRegression(), False),
    ('RF pequeño',   RandomForestRegressor(n_estimators=50, max_depth=8, random_state=SEED, n_jobs=-1), False),
    ('RF grande',    RandomForestRegressor(n_estimators=300, max_depth=20, random_state=SEED, n_jobs=-1), False),
    ('XGB rápido',   XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=SEED, n_jobs=-1), False),
    ('XGB grande',   XGBRegressor(n_estimators=500, max_depth=8, learning_rate=0.05, random_state=SEED, n_jobs=-1), False),
    ('MLP pequeño',  MLPRegressor(hidden_layer_sizes=(32,), max_iter=100, random_state=SEED, early_stopping=True), True),
    ('MLP grande',   MLPRegressor(hidden_layer_sizes=(128,64,32), max_iter=200, random_state=SEED, early_stopping=True), True),
]

resultados_pareto = []
for nombre, modelo, necesita_escala in configs:
    Xtr = X_tr_w_sc if necesita_escala else X_tr_w
    Xte = X_te_w_sc if necesita_escala else X_te_w
    t0 = time.time()
    modelo.fit(Xtr, y_tr)
    t_ent = time.time() - t0
    r2_te = r2_score(y_te, modelo.predict(Xte))
    resultados_pareto.append({'Modelo': nombre, 'Tiempo (s)': t_ent, 'R² test': r2_te})

df_pareto = pd.DataFrame(resultados_pareto)
df_pareto.round(3)

In [ ]:
# Identificamos frontera de Pareto (mejor R² para cada tiempo, o menor tiempo para cada R²)
df_sorted = df_pareto.sort_values('Tiempo (s)').reset_index(drop=True)
pareto_mask = []
best_r2 = -np.inf
for _, row in df_sorted.iterrows():
    if row['R² test'] > best_r2:
        pareto_mask.append(True)
        best_r2 = row['R² test']
    else:
        pareto_mask.append(False)
df_sorted['Pareto'] = pareto_mask

plt.figure(figsize=(9, 6))
for i, row in df_sorted.iterrows():
    color = 'crimson' if row['Pareto'] else 'steelblue'
    marker = '*' if row['Pareto'] else 'o'
    size = 300 if row['Pareto'] else 150
    plt.scatter(row['Tiempo (s)'], row['R² test'], s=size, c=color, marker=marker,
                edgecolors='black', linewidth=1.2, alpha=0.85, zorder=3)
    plt.annotate(row['Modelo'], (row['Tiempo (s)'], row['R² test']),
                 textcoords="offset points", xytext=(8, 8), fontsize=9)

# Línea de frontera
pareto_pts = df_sorted[df_sorted['Pareto']]
plt.plot(pareto_pts['Tiempo (s)'], pareto_pts['R² test'], '--', color='crimson', alpha=0.5, zorder=1)

plt.xlabel('Tiempo de entrenamiento (s)'); plt.ylabel('R² test')
plt.title('Frontera de Pareto: Tiempo vs Desempeño\n(⭐ = modelos óptimos)')
plt.xscale('log'); plt.grid(True); plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio 4

- Los modelos en la **frontera de Pareto** (⭐) son los que ofrecen el mejor trade-off: no hay ningún otro modelo que sea a la vez más rápido **y** mejor.
- Los modelos que quedan por debajo de la frontera son **dominados**: existe otro modelo estrictamente mejor en al menos una dimensión sin ser peor en la otra.
- La **regresión lineal** suele estar en la frontera por ser extremadamente rápida — aunque su desempeño sea modesto.
- **Regla práctica:** en producción, si dos modelos tienen desempeño similar en test, el más rápido gana (menor costo de inferencia y reentrenamiento).


## 🎁 Ejercicio Bonus · Curvas de aprendizaje con distintos tamaños de datos

### 📝 Enunciado
¿Qué pasa si tenemos poca data? Comparar cómo se comportan los modelos con 10%, 25%, 50%, 75% y 100% del train.

### 💭 Hipótesis
- Los modelos más complejos (redes neuronales, XGBoost grande) necesitan más datos para brillar.
- Con poca data, un modelo simple puede superar a uno complejo.


In [ ]:
fracciones = [0.1, 0.25, 0.5, 0.75, 1.0]
modelos_bonus = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=15, random_state=SEED, n_jobs=-1),
    'XGBoost':           XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=SEED, n_jobs=-1),
    'MLP':               MLPRegressor(hidden_layer_sizes=(64,32), max_iter=200, random_state=SEED, early_stopping=True),
}

curvas_datos = {n: [] for n in modelos_bonus}
n_train = len(X_tr_w)

for frac in fracciones:
    n_sub = int(frac * n_train)
    idx = np.random.RandomState(SEED).choice(n_train, n_sub, replace=False)
    X_sub, y_sub = X_tr_w[idx], y_tr[idx]
    X_sub_sc, y_sub_sc = X_tr_w_sc[idx], y_tr[idx]
    
    for nombre, modelo in modelos_bonus.items():
        # Clonamos el modelo para no reutilizar estado
        from sklearn.base import clone
        m = clone(modelo)
        if nombre == 'MLP' or nombre == 'Linear Regression':
            m.fit(X_sub_sc, y_sub_sc)
            r2 = r2_score(y_te, m.predict(X_te_w_sc))
        else:
            m.fit(X_sub, y_sub)
            r2 = r2_score(y_te, m.predict(X_te_w))
        curvas_datos[nombre].append(r2)

plt.figure(figsize=(10, 6))
for nombre, r2s in curvas_datos.items():
    plt.plot([int(f * n_train) for f in fracciones], r2s, 'o-', linewidth=2, markersize=8, label=nombre)
plt.xlabel('Tamaño de entrenamiento (# muestras)'); plt.ylabel('R² en test')
plt.title('Curvas de aprendizaje: R² vs cantidad de datos')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

### ✅ Conclusión del Ejercicio Bonus

- Con **poca data** (10%), Random Forest y XGBoost tienden a superar al MLP porque son menos "hambrientos" de datos.
- El **MLP mejora más** con cada incremento de datos — su pendiente de aprendizaje es más pronunciada.
- La regresión lineal **satura** rápido: más datos no la ayudan porque su capacidad es limitada.
- Si tu curva de aprendizaje sigue subiendo al final → conseguir más datos ayudará. Si es plana → tu limitación está en el modelo, no en los datos.


## 📋 Resumen de aprendizajes

| Ejercicio | Conclusión clave |
|---|---|
| 1. Outliers | Winsorizar ayuda más a modelos lineales y NN que a árboles |
| 2. Sobreajuste | La brecha train−test es el mejor detector visual |
| 3. Complejidad NN | Existe un techo — redes más grandes no siempre ayudan |
| 4. Tiempo vs desempeño | Solo los modelos en la frontera de Pareto son óptimos |
| Bonus. Cantidad de datos | Los modelos complejos necesitan más datos para brillar |

### 💭 Preguntas de discusión final

1. Si en tu proyecto solo tienes **500 muestras**, ¿qué modelo elegirías y por qué?
2. Si el costo computacional te lo permite, ¿tiene sentido siempre probar XGBoost antes que una red neuronal? Justifica.
3. En un contexto de **producción con reentrenamientos diarios**, ¿cómo cambia tu criterio de elección?
